# Import libraries

In [1]:
import pandas as pd
import numpy as np

from sklearn.metrics.pairwise import cosine_similarity

In [4]:
ratings = pd.read_csv(
    "ml-100k/u.data",
    sep="\t",
    names=["user_id","movie_id","rating","timestamp"]
)

movies = pd.read_csv(
    "ml-100k/u.item",
    sep="|",
    encoding="latin-1",
    header=None
)

movies = movies[[0,1]]
movies.columns = ["movie_id","title"]

print(ratings.head())
print(movies.head())

   user_id  movie_id  rating  timestamp
0      196       242       3  881250949
1      186       302       3  891717742
2       22       377       1  878887116
3      244        51       2  880606923
4      166       346       1  886397596
   movie_id              title
0         1   Toy Story (1995)
1         2   GoldenEye (1995)
2         3  Four Rooms (1995)
3         4  Get Shorty (1995)
4         5     Copycat (1995)


# Merge movie titles with ratings

In [5]:
data = pd.merge(ratings, movies, on="movie_id")

print(data.head())

   user_id  movie_id  rating  timestamp                       title
0      196       242       3  881250949                Kolya (1996)
1      186       302       3  891717742    L.A. Confidential (1997)
2       22       377       1  878887116         Heavyweights (1994)
3      244        51       2  880606923  Legends of the Fall (1994)
4      166       346       1  886397596         Jackie Brown (1997)


#  Create user-item matrix

Rows = users

Columns = movies

Values = ratings

In [8]:
user_item_matrix = data.pivot_table(
    index="user_id",
    columns="title",
    values="rating"
)

print(user_item_matrix.head())

title    'Til There Was You (1997)  1-900 (1994)  101 Dalmatians (1996)  \
user_id                                                                   
1                              NaN           NaN                    2.0   
2                              NaN           NaN                    NaN   
3                              NaN           NaN                    NaN   
4                              NaN           NaN                    NaN   
5                              NaN           NaN                    2.0   

title    12 Angry Men (1957)  187 (1997)  2 Days in the Valley (1996)  \
user_id                                                                 
1                        5.0         NaN                          NaN   
2                        NaN         NaN                          NaN   
3                        NaN         2.0                          NaN   
4                        NaN         NaN                          NaN   
5                        NaN        

# Replace missing ratings with 0

In [9]:
user_item_matrix = user_item_matrix.fillna(0)

# cosine similarity

In [10]:
user_similarity = cosine_similarity(user_item_matrix)

user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_item_matrix.index,
    columns=user_item_matrix.index
)

print(user_similarity_df.head())

user_id       1         2         3         4         5         6         7    \
user_id                                                                         
1        1.000000  0.168937  0.048388  0.064561  0.379670  0.429682  0.443097   
2        0.168937  1.000000  0.113393  0.179694  0.073623  0.242106  0.108604   
3        0.048388  0.113393  1.000000  0.349781  0.021592  0.074018  0.067423   
4        0.064561  0.179694  0.349781  1.000000  0.031804  0.068431  0.091507   
5        0.379670  0.073623  0.021592  0.031804  1.000000  0.238636  0.374733   

user_id       8         9         10   ...       934       935       936  \
user_id                                ...                                 
1        0.320079  0.078385  0.377733  ...  0.372213  0.119860  0.269860   
2        0.104257  0.162470  0.161273  ...  0.147095  0.310661  0.363328   
3        0.084419  0.062039  0.066217  ...  0.033885  0.043453  0.167140   
4        0.188060  0.101284  0.060859  ...  0.054615

# Function to recommend movies for a user

In [11]:
def recommend_movies(user_id, num_recommendations=5):

    similar_users = user_similarity_df[user_id].sort_values(ascending=False)

    similar_users = similar_users.iloc[1:11]

    user_ratings = user_item_matrix.loc[user_id]

    unseen_movies = user_ratings[user_ratings == 0].index

    scores = {}

    for movie in unseen_movies:

        movie_ratings = user_item_matrix[movie]

        score = np.dot(similar_users, movie_ratings[similar_users.index])

        scores[movie] = score

    recommendations = sorted(scores.items(), key=lambda x: x[1], reverse=True)

    return recommendations[:num_recommendations]

# Test recommendation system

In [12]:
recommendations = recommend_movies(user_id=1)

for movie, score in recommendations:
    print(movie)

Schindler's List (1993)
Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb (1963)
Stand by Me (1986)
E.T. the Extra-Terrestrial (1982)
Batman (1989)


# Evaluation using Precision

In [13]:
def precision_at_k(user_id, k=5):

    recommended = recommend_movies(user_id, k)

    recommended_movies = [movie for movie,score in recommended]

    actual_ratings = data[data["user_id"] == user_id]

    liked_movies = actual_ratings[actual_ratings["rating"] >= 4]["title"]

    relevant = len(set(recommended_movies) & set(liked_movies))

    precision = relevant / k

    return precision

In [14]:
print("Precision@5:", precision_at_k(1,5))

Precision@5: 0.0


# similarity between movies.

In [15]:
item_similarity = cosine_similarity(user_item_matrix.T)

item_similarity_df = pd.DataFrame(
    item_similarity,
    index=user_item_matrix.columns,
    columns=user_item_matrix.columns
)

# Recommendation using item similarity

In [16]:
def item_based_recommend(movie_name, top_n=5):

    similar_movies = item_similarity_df[movie_name].sort_values(ascending=False)

    return similar_movies.iloc[1:top_n+1]

# such as 

In [17]:
print(item_based_recommend("Star Wars (1977)"))

title
Return of the Jedi (1983)          0.884476
Raiders of the Lost Ark (1981)     0.764885
Empire Strikes Back, The (1980)    0.749819
Toy Story (1995)                   0.734572
Godfather, The (1972)              0.697332
Name: Star Wars (1977), dtype: float64


# Matrix Factorization (SVD)

In [18]:
from sklearn.decomposition import TruncatedSVD

svd = TruncatedSVD(n_components=20)

matrix_reduced = svd.fit_transform(user_item_matrix)

reconstructed = np.dot(matrix_reduced, svd.components_)

# Create prediction matrix

In [19]:
predicted_ratings = pd.DataFrame(
    reconstructed,
    index=user_item_matrix.index,
    columns=user_item_matrix.columns
)

print(predicted_ratings.head())

title    'Til There Was You (1997)  1-900 (1994)  101 Dalmatians (1996)  \
user_id                                                                   
1                         0.009221      0.035395               0.113631   
2                         0.102753      0.008565               0.190549   
3                         0.053332      0.004987              -0.160660   
4                        -0.023238     -0.013939              -0.091746   
5                        -0.068689      0.066481               0.870647   

title    12 Angry Men (1957)  187 (1997)  2 Days in the Valley (1996)  \
user_id                                                                 
1                   0.704958    0.140934                     1.153230   
2                   0.102358    0.162010                     0.535734   
3                   0.083924    0.612654                     0.194850   
4                   0.237433    0.306256                    -0.145193   
5                   0.086499   -0.23